<a href="https://colab.research.google.com/github/nitin-rajesh/NLP-RL-Pipeline/blob/main/NaturalInstructionsJudge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import random
from ast import literal_eval

from datasets import Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from huggingface_hub import login
from google.colab import userdata
# userdata.get('secretName')
# load from Colab Secrets
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

# ensure all HF libs use this token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
os.environ["HF_TOKEN"] = hf_token

print("HF auth complete.")


HF auth complete.


In [ ]:
# save as generate_prompts_pipeline.py
import json
import os
import time
from datasets import load_dataset
from tqdm.auto import tqdm
import hashlib

# --- CONFIG ---
DATASET_NAME = "Muennighoff/natural-instructions"
TARGET_COUNT = 1_000_000
SEED = 42
OUTPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgen.jsonl"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  # Changed to an accessible model
MAX_NEW_TOKENS = 32000
TEMPERATURE = 0.0
TOP_P = 1.0
BATCH_SIZE = 1  # we do 1 per call to keep stateless, change if your server supports batch
SAVE_EVERY = 100  # flush to disk every N examples (we append per example anyway)
# ----------------

# Utility: safe JSON extraction from model text (tries to find a top-level JSON object)
def extract_json_from_text(text):
    text = text.strip()
    # Try direct JSON first
    try:
        return json.loads(text)
    except Exception:
        pass
    # Otherwise find first '{' and last '}' and try
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            pass
    # fallback: return None
    return None

# Create a deterministic unique key for deduping
def dedupe_key(row):
    # prefer id if present
    if "id" in row and row["id"] not in (None, ""):
        return f"id::{row['id']}"
    # fall back to input (some rows use "input" or "inputs")
    input_field = row.get("input") or row.get("inputs") or ""
    if input_field is None:
        input_field = ""
    # hash to reduce memory footprint
    h = hashlib.sha256(input_field.encode("utf-8")).hexdigest()
    return f"input_hash::{h}"

In [ ]:
# Build the instruction given a dataset row
def build_generation_system_and_user(definition, input_text):

    system = (
    "You are a professional prompt engineer. For the given task definition and input, "
    "use your world knowledge and editorial judgment to rewrite the task definition into a clearer, "
    "more explicit, more actionable instruction that will yield a high-quality answer from a downstream LLM. "
    "The rewritten instruction must fully replace the original task definition and must be embedded directly inside each prompt.\n\n"

    "Create exactly 3 distinct candidate prompts that will each be fed directly to another LLM. "
    "Each prompt must be self-contained and include:\n"
    "  - the rewritten version of the task definition (as a single integrated instruction),\n"
    "  - guidance or constraints that would help the downstream LLM produce a strong answer,\n"
    "  - and the placeholder or directive for how the downstream model should use the provided input.\n\n"

    "Output ONLY valid JSON with the schema:\n"
    '{ \"prompts\": [ '
    '{\"id\":\"p1\",\"prompt\":\"...\"}, '
    '{\"id\":\"p2\",\"prompt\":\"...\"}, '
    '{\"id\":\"p3\",\"prompt\":\"...\"} '
    '] }\n\n'

    "STRICT RULES:\n"
    "1. Do NOT output anything outside the JSON object.\n"
    "2. The original task definition must NOT appear in the prompts. Only your improved rewritten version should appear Each prompt must reflect your improved version of the task definition—add clarity, remove ambiguity, "
    "and include any constraints, assumptions, or world-knowledge context that improve answer quality.\n"
    "3. Each 'prompt' must be ready to feed directly to a downstream LLM as-is.\n"
    "4. Make the three prompts meaningfully different:\n"
    "   - p1: concise, sharpened, high-precision instruction.\n"
    "   - p2: structured or step-by-step instruction that guides reasoning.\n"
    "   - p3: enriched instruction that incorporates relevant world knowledge, assumptions, or clarifications.\n"
    "5. Include explicit output format requirements inside the prompts when helpful (e.g., JSON schema, bullet list, single sentence, etc.).\n"
    "6. Remeber your task is to improve on the task definition.\n\n"

    "Only output the JSON object."
  )


    user = f"Task definition:\n{definition}\n\nInput:\n{input_text}\n\nProduce JSON now."
    return system, user


In [ ]:
import json
from typing import Tuple, Dict, Optional

def build_generation_system_and_user(
    record: Dict,
    few_shot_examples: Optional[list] = None
) -> Tuple[str, str]:
    """
    Build (system_text, user_text) for the judge model from a JSONL record.

    This version matches the output record schema you use in your pipeline:
        - definition_orig / inference_orig
        - definition_p1  / inference_p1
        - definition_p2  / inference_p2
        - definition_p3  / inference_p3

    Args:
      - record: dict parsed from the JSONL line. Expected (or fallback) keys:
          example_id, input_text,
          orig_prompt or definition_orig or orig, inference_orig,
          p1, inference_p1, p2, inference_p2, p3, inference_p3
      - few_shot_examples: optional list of (record_dict, desired_output_dict) pairs to include
                           as in-context examples (they will be serialized into the user prompt).

    Returns:
      (system_text, user_text)
    """

    # Normalize fields to match your rec_out schema
    example_id = record.get("example_id") or record.get("id") or ""
    input_text = record.get("input_text") or record.get("input") or record.get("text") or ""

    definition_orig = (
        record.get("orig_prompt")
        or record.get("definition_orig")
        or record.get("orig")
        or ""
    )
    inference_orig = record.get("inference_orig") or record.get("inference") or ""

    definition_p1 = record.get("p1") or record.get("definition_p1") or ""
    inference_p1 = record.get("inference_p1") or ""

    definition_p2 = record.get("p2") or record.get("definition_p2") or ""
    inference_p2 = record.get("inference_p2") or ""

    definition_p3 = record.get("p3") or record.get("definition_p3") or ""
    inference_p3 = record.get("inference_p3") or ""

    sources = record.get("sources", None)
    processed_at = record.get("processed_at", None)

    # System prompt: explicit judge instructions (strict JSON-only output)
    system_text = (
        "JUDGE SYSTEM INSTRUCTIONS:\n\n"
        "Context: A human user originally provided an instruction (definition_orig / orig_prompt)\n"
        "and applied it to a piece of text (input_text), producing inference_orig. You (the judge)\n"
        "also have three alternative instructions (definition_p1, definition_p2, definition_p3)\n"
        "and their produced answers (inference_p1..inference_p3).\n\n"
        "Task: Given the ORIGINAL INSTRUCTION (definition_orig) and the same input_text, determine\n"
        "which of the four produced answers (inference_orig, inference_p1, inference_p2, inference_p3)\n"
        "a human would PREFER. Evaluate the *answers* (not the instructions) on these dimensions:\n"
        "  - Usefulness (does the answer help accomplish the user's intent?),\n"
        "  - Informativeness (adds relevant useful details),\n"
        "  - Instruction-following (how well does it follow the ORIGINAL INSTRUCTION?),\n"
        "  - Accuracy (is it plausible/correct given input_text?),\n"
        "  - Clarity & readability,\n"

        "  - Human-friendliness/tone.\n\n"
        "OUTPUT FORMAT (MUST be followed EXACTLY): Output ONE valid JSON object and NOTHING else.\n"
        "The JSON MUST match this schema exactly:\n"
        "{\n"
        '  "example_id": string,\n'
        '  "rank_scores": {"orig": int, "p1": int, "p2": int, "p3": int},\n'
        '  "labels": {"orig": "good|neutral|bad", "p1": "good|neutral|bad", "p2": "good|neutral|bad", "p3": "good|neutral|bad"},\n'
        '  "selected_good": string,\n'
        '  "selected_bad": string,\n'
        '  "selected_neutral": [string, string],\n'
        '  "short_reasons": {"orig": string, "p1": string, "p2": string, "p3": string}\n'
        "}\n\n"
        "STRICT CONSTRAINTS:\n"
        "- Exactly one label must be 'good', exactly one 'bad', and exactly two 'neutral'.\n"
        "- Rank scores MUST be the integers 1,2,3,4 with NO ties (1 = most preferred).\n"
        "- Each short reason must be concise (<= ~50 words). on why you gave it that ranking.\n"
        "- If an inference is empty/missing, treat it as a negative signal but weigh other\n"
        "  dimensions as appropriate.\n"
        "- Use internal reasoning and world knowledge to judge, but DO NOT reveal chain-of-thought.\n"
        "- Output only the single JSON object; do not include any extra commentary or explanation.\n\n"
        "Now read the RECORD_JSON provided in the USER message and output the single JSON judgment object."
    )

    # Build the user payload following your rec_out structure (compact)
    user_payload = {
        "example_id": example_id,
        "input_text": input_text,

        "definition_orig": definition_orig,
        "inference_orig": inference_orig,

        "definition_p1": definition_p1,
        "inference_p1": inference_p1,

        "definition_p2": definition_p2,
        "inference_p2": inference_p2,

        "definition_p3": definition_p3,
        "inference_p3": inference_p3,
    }

    # Optionally include sources / processed_at if present (for context only)
    if sources is not None:
        user_payload["sources"] = sources
    if processed_at is not None:
        user_payload["processed_at"] = processed_at

    # Build user_text parts and optionally include few-shot examples
    user_text_parts = []
    if few_shot_examples:
        user_text_parts.append("FEW-SHOT EXAMPLES (for guidance; do NOT copy them into your answer):")
        for ex_idx, (ex_rec, ex_out) in enumerate(few_shot_examples, start=1):
            ex_payload = {
                "example_id": ex_rec.get("example_id", f"ex{ex_idx}"),
                "input_text": ex_rec.get("input_text", "")[:1000],
                "definition_orig": ex_rec.get("orig_prompt") or ex_rec.get("definition_orig") or ex_rec.get("orig", ""),
                "inference_orig": ex_rec.get("inference_orig", ""),
                "definition_p1": ex_rec.get("p1", ""),
                "inference_p1": ex_rec.get("inference_p1", ""),
                "definition_p2": ex_rec.get("p2", ""),
                "inference_p2": ex_rec.get("inference_p2", ""),
                "definition_p3": ex_rec.get("p3", ""),
                "inference_p3": ex_rec.get("inference_p3", ""),
            }
            user_text_parts.append(f"EXAMPLE_RECORD_{ex_idx}:\n{json.dumps(ex_payload, ensure_ascii=False)}")
            user_text_parts.append(f"EXAMPLE_OUTPUT_{ex_idx}:\n{json.dumps(ex_out, ensure_ascii=False)}")

    user_text_parts.append("RECORD_JSON:\n" + json.dumps(user_payload, ensure_ascii=False))
    user_text_parts.append("\n\nOutput ONLY the single JSON judgment object as specified in the system instructions.")

    user_text = "\n\n".join(user_text_parts)

    return system_text, user_text


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# global single-load cache
_LOCAL_GEN = {"ready": False}

def generate_with_local_model(prompt_text,
                              model_name="mistralai/Mistral-7B-Instruct-v0.3",
                              max_new_tokens=256,
                              temperature=0.0,
                              top_p=1.0):
    """
    MINIMAL version.
    - Loads model ONCE, in FP16 full weights.
    - No try/except anywhere.
    - If anything fails (download, HF token, OOM, missing lib)
      → Python will show the real error.
    """

    global _LOCAL_GEN

    if not _LOCAL_GEN["ready"]:
        # ---------- Load tokenizer ----------
        tokenizer = AutoTokenizer.from_pretrained(
            model_name
        )
        # ensure tokenizer has a pad token
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id


        # ---------- Load model in FP16 (no fallback) ----------
        print(f"[model] Loading {model_name} in FP16...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",             # puts layers on your GPU
            trust_remote_code=True,
            use_safetensors=True
        )
        print("[model] Loaded successfully.")

        model.eval()

        _LOCAL_GEN["tokenizer"] = tokenizer
        _LOCAL_GEN["model"] = model
        _LOCAL_GEN["ready"] = True

    # ---------- Generate ----------
    tokenizer = _LOCAL_GEN["tokenizer"]
    model = _LOCAL_GEN["model"]

    inputs = tokenizer(prompt_text, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0.0),
        temperature=temperature,
        top_p=top_p,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove echoed prompt if present
    if text.startswith(prompt_text):
        text = text[len(prompt_text):].strip()

    return text


In [ ]:
# definition = "Summarize the input text."
# input_text = "Alice went to the market and bought apples."

# system_txt, user_txt = build_generation_system_and_user(definition, input_text)
# combined_prompt = system_txt + "\n\n" + user_txt
# # combined_prompt = f"<s>[INST] {system_txt}\n\n{user_txt} [/INST]"

# print(combined_prompt)
# print("\n=== Running Local Pilot ===")
# out = generate_with_local_model(
#     combined_prompt,
#     max_new_tokens=128,
#     temperature=0.0
# )
# print("\nModel Output:\n", out[:1000])


In [ ]:
!pip install -U datasets --quiet



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 49.1 MB/s eta 0:00:00


In [ ]:
# stream_sample_natural_instructions.py
# Copy-paste into Colab and run. Requires `pip install -U datasets tqdm` if not already installed.

import json, hashlib, random, time
from itertools import islice
from datasets import load_dataset
from tqdm.auto import tqdm

# ---------- CONFIG ----------
DATASET_NAME = "Muennighoff/natural-instructions"
# OUTPUT_JSONL = "natural_instructions_1M_streamed.jsonl"
TARGET_COUNT = 1_000_000
SHUFFLE_BUFFER = 50_000    # memory/time tradeoff: bigger -> better shuffle
SEED = 42
WRITE_FLUSH_EVERY = 1000   # flush to disk every N writes
# ----------------------------

def dedupe_key(row):
    # prefer id if present; else hash the input text
    if row.get("id"):
        return f"id::{row['id']}"
    inp = row.get("input") or row.get("inputs") or ""
    return "input_hash::" + hashlib.sha256((inp or "").encode("utf-8")).hexdigest()

def stream_and_write():
    print("Streaming dataset (no splits created/prepared). This avoids split creation errors.")
    ds_stream = load_dataset(DATASET_NAME, split="train", streaming=True)  # streaming=True avoids local split creation
    rng = random.Random(SEED)

    seen = set()
    buffer = []
    written = 0
    start = time.time()

    with open(OUTPUT_JSONL, "w", encoding="utf-8") as fout:
        for row in tqdm(ds_stream):
            # dedupe by id / input
            key = dedupe_key(row)
            if key in seen:
                continue
            seen.add(key)

            buffer.append(row)

            # When buffer is large enough, pop a random element and write it out:
            if len(buffer) >= SHUFFLE_BUFFER:
                i = rng.randrange(len(buffer))
                sample = buffer.pop(i)
                fout.write(json.dumps(sample, ensure_ascii=False) + "\n")
                written += 1
                if written % WRITE_FLUSH_EVERY == 0:
                    fout.flush()
                if written % 10000 == 0:
                    elapsed = time.time() - start
                    print(f"Written {written} (elapsed {elapsed:.1f}s). Buffer size {len(buffer)}.")
                if written >= TARGET_COUNT:
                    break

        # Flush remaining buffer in random order until we hit TARGET_COUNT
        rng.shuffle(buffer)
        for sample in buffer:
            if written >= TARGET_COUNT:
                break
            fout.write(json.dumps(sample, ensure_ascii=False) + "\n")
            written += 1
            if written % WRITE_FLUSH_EVERY == 0:
                fout.flush()

    print(f"Done. Wrote {written} examples to {OUTPUT_JSONL}")

# if __name__ == "__main__":
#     stream_and_write()


In [ ]:
# # Replace genprompts() with this batched version

# import torch
# from itertools import islice

INPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_inference_qwen_generate3.jsonl"   # your existing jsonl
OUTPUT_JSONL = "/content/drive/MyDrive/AdvNLP/judgemistral1.jsonl" # new file to store prompt-generation outputs
TARGET_COUNT = 1_000_000   # how many examples to process (set lower for a pilot)
SAVE_EVERY = 100
# Batching configuration
BATCH_SIZE = 4            # start with 8; increase to 16 if memory allows
MAX_NEW_TOKENS = 32000         # per-example generation length (adjust if needed)
TEMPERATURE = 0.3
TOP_P = 0.9



In [ ]:


# Put these tuning vars near the top of your script
PER_BATCH_MAX_SECONDS = 120   # how long to allow a batch generation (tweak as needed)
MAX_NEW_TOKENS = 4000         # safer default; keep small for prompt-generation
RECORDS_PER_BATCH = 8
# BATCH_SIZE = 8               # keep as you had it

# Replacement flush_batch() — drop-in
import traceback, torch

In [ ]:
import os
import json
import time
import traceback
from tqdm import tqdm

def genprompts(
    input_jsonl=INPUT_JSONL,
    output_jsonl=OUTPUT_JSONL,
    records_per_batch=RECORDS_PER_BATCH,
    target_count=None,
):
    """
    Resumable batched generation. Writes minimal out_record with only:
      - example_id
      - generation (model, params, raw_text or error, parsed_json if available)
      - processed_at
    """

    # sanity checks
    if not os.path.exists(input_jsonl):
        raise FileNotFoundError(f"Input JSONL not found: {input_jsonl}")

    print(f"Resuming/starting batched prompt generation.\nInput: {input_jsonl}\nOutput: {output_jsonl}")
    processed_ids = read_processed_ids(output_jsonl)
    print(f"Already processed examples found: {len(processed_ids)}")

    os.makedirs(os.path.dirname(output_jsonl), exist_ok=True)
    out_f = open(output_jsonl, "a", encoding="utf-8")  # append mode for resumability
    processed = 0
    start_time = time.time()
    batch_counter = 0

    # Ensure local model/tokenizer is loaded (lazy init)
    if "_LOCAL_GEN" not in globals() or not _LOCAL_GEN.get("ready", False):
        try:
            _ = generate_with_local_model("Warm up", max_new_tokens=1, temperature=0.0)
        except Exception:
            pass

    tokenizer = _LOCAL_GEN["tokenizer"]
    model = _LOCAL_GEN["model"]
    device = _LOCAL_GEN.get("device", "cuda" if torch.cuda.is_available() else "cpu")

    # Count file lines (optional)
    try:
        with open(input_jsonl, "r", encoding="utf-8") as fcount:
            total_lines = sum(1 for _ in fcount)
    except Exception:
        total_lines = None

    batch_rows = []
    batch_meta = []

    def flush_batch():
        nonlocal processed, batch_rows, batch_meta, batch_counter
        if not batch_rows:
            return

        batch_counter += 1
        ids_sample = [m['example_id'] for m in batch_meta[:min(5, len(batch_meta))]]
        print(f"\n[batch {batch_counter}] START - size={len(batch_rows)}. Example IDs: {ids_sample} (showing up to 5)")

        t0 = time.time()
        try:
            # Build combined prompts for the batch
            combined_prompts = []
            for row, meta in zip(batch_rows, batch_meta):
              sys_txt, user_txt = build_generation_system_and_user(row)
              combined_prompts.append(sys_txt + "\n\n" + user_txt)


            # Tokenize the batch (pad to longest)
            t_tok0 = time.time()
            tokenized = tokenizer(combined_prompts, return_tensors="pt", padding=True, truncation=True)
            t_tok1 = time.time()

            if device == "cuda":
                tokenized = {k: v.cuda() for k, v in tokenized.items()}

            input_len = tokenized["input_ids"].shape[1]
            ctx_limit = getattr(model.config, "max_position_embeddings", None)
            if ctx_limit is not None:
                dyn_max_new = max(1, min(MAX_NEW_TOKENS, ctx_limit - input_len))
            else:
                dyn_max_new = min(MAX_NEW_TOKENS, 1024)  # fallback safe cap

            print(f"[batch {batch_counter}] tokenized len={input_len}, dyn_max_new_tokens={dyn_max_new}, tokenization_time={(t_tok1-t_tok0):.3f}s")

            # Prepare generate kwargs:
            do_sample = (TEMPERATURE > 0.0)
            gen_kwargs = dict(
                **tokenized,
                max_new_tokens=dyn_max_new,
                do_sample=do_sample,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                max_time=PER_BATCH_MAX_SECONDS,   # stops generation after this many seconds (transformers)
            )
            if do_sample:
                gen_kwargs["temperature"] = float(TEMPERATURE)
                gen_kwargs["top_p"] = float(TOP_P)

            # Synchronize and run generation with explicit timing
            torch.cuda.empty_cache()
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            gen_start = time.time()
            with torch.no_grad():
                outputs = model.generate(**gen_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            gen_end = time.time()

            # Decode outputs (batch)
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

            # Write each example (minimal out_record)
            written = 0
            for meta, prompt_text, raw_out in zip(batch_meta, combined_prompts, decoded):
                example_id = meta["example_id"]
                gen_text = raw_out
                if gen_text.startswith(prompt_text):
                    gen_text = gen_text[len(prompt_text):].strip()

                parsed = extract_json_from_text(gen_text)
                # Build minimal generation object
                generation_obj = {
                    "model": MODEL_NAME,
                    "params": {
                        "batch_size": len(batch_rows),
                        "max_new_tokens": dyn_max_new,
                        "temperature": float(TEMPERATURE),
                        "top_p": float(TOP_P),
                    },
                    "raw_text": gen_text,
                }
                if parsed is not None:
                    generation_obj["parsed_json"] = parsed
                else:
                    generation_obj["error_parse"] = True

                out_record = {
                    "example_id": example_id,
                    "generation": generation_obj,
                    "processed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                }

                out_f.write(json.dumps(out_record, ensure_ascii=False) + "\n")
                processed += 1
                processed_ids.add(example_id)
                written += 1

            out_f.flush()
            t1 = time.time()

            # GPU stats
            try:
                reserved_gb = torch.cuda.memory_reserved() / 1024**3
                allocated_gb = torch.cuda.memory_allocated() / 1024**3
            except Exception:
                reserved_gb = allocated_gb = 0.0

            print(f"[batch {batch_counter}] DONE - total={(t1-t0):.2f}s gen={(gen_end-gen_start):.2f}s token_time={(t_tok1-t_tok0):.3f}s written={written} processed_total={processed}")
            print(f"[batch {batch_counter}] GPU reserved={reserved_gb:.2f}GB allocated={allocated_gb:.2f}GB")

        except Exception as e:
            tb = traceback.format_exc()
            print(f"[batch {batch_counter}] ERROR during generation: {e}\n{tb}")
            # write a minimal error record for each example so we don't stall forever
            for meta in batch_meta:
                example_id = meta["example_id"]
                generation_obj = {
                    "model": MODEL_NAME,
                    "params": {"batch_size": len(batch_rows)},
                    "raw_text": "",
                    "error": str(e),
                }
                rec = {
                    "example_id": example_id,
                    "generation": generation_obj,
                    "processed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                }
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                processed_ids.add(example_id)
            out_f.flush()
        finally:
            # always clear buffers so loop continues
            batch_rows = []
            batch_meta = []

    # Iterate input file and collect batches
    try:
        with open(input_jsonl, "r", encoding="utf-8") as fin:
            for idx, raw in enumerate(tqdm(fin, total=total_lines)):
                if processed >= target_count if target_count is not None else False:
                    break

                try:
                    row = json.loads(raw)
                except Exception:
                    print(f"Skipping malformed line idx={idx}")
                    continue

                example_id = row.get("example_id") or row.get("id") or f"row_{idx}"
                if example_id in processed_ids:
                    continue

                definition = row.get("definition", "") or ""
                input_text = row.get("input", "") or row.get("inputs", "") or row.get("text", "") or ""

                batch_rows.append(row)
                batch_meta.append({"example_id": example_id, "source_idx": idx, "definition": definition, "input_text": input_text})

                if len(batch_rows) >= records_per_batch:
                    flush_batch()

                # periodic flush for long runs
                if processed % SAVE_EVERY == 0 and processed > 0:
                    elapsed = time.time() - start_time
                    try:
                        gpu_reserved = torch.cuda.memory_reserved() / 1024**3
                    except Exception:
                        gpu_reserved = 0.0
                    print(f"Processed {processed} new examples (elapsed {elapsed:.1f}s). GPU reserved: {gpu_reserved:.2f}GB")

            # flush any remaining rows
            if batch_rows:
                flush_batch()

    finally:
        out_f.close()

    elapsed = time.time() - start_time
    print(f"Done. New processed examples in this run: {processed}. Output file: {output_jsonl} elapsed={elapsed:.1f}s")
    return processed

# Example usage:
# genprompts(input_jsonl=INPUT_JSONL, output_jsonl=OUTPUT_JSONL, records_per_batch=RECORDS_PER_BATCH, target_count=TARGET_COUNT)


In [ ]:
if __name__ == "__main__":
    genprompts()

Resuming/starting batched prompt generation.
Input: /content/drive/MyDrive/AdvNLP/natural_instructions_1M_inference_qwen_generate3.jsonl
Output: /content/drive/MyDrive/AdvNLP/judgemistral1.jsonl
Already processed examples found: 4393


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

100%|█████████▉| 5497/5511 [1:06:57<00:48,  3.49s/it]

[batch 138] DONE - total=30.21s gen=30.17s token_time=0.018s written=8 processed_total=1104
[batch 138] GPU reserved=18.21GB allocated=13.51GB

[batch 139] START - size=8. Example IDs: ['task046-3c87f2ddad9e4c928ae323950f19a627', 'task027-a98fe50a4db84bd3bffce9e39f26ea84', 'task024-8b1f09d2086f4c4e938ab58583354588', 'task046-4bfdc270000c404895ca95e507b99438', 'task027-243f0d482ad3415f8f140468f05d77da'] (showing up to 5)
[batch 139] tokenized len=1986, dyn_max_new_tokens=4000, tokenization_time=0.015s


100%|██████████| 5511/5511 [1:07:18<00:00,  1.36it/s]

[batch 139] DONE - total=21.20s gen=21.17s token_time=0.015s written=8 processed_total=1112
[batch 139] GPU reserved=18.04GB allocated=13.51GB

[batch 140] START - size=6. Example IDs: ['task027-3efde64e8c28483db894b7b27fc189b5', 'task028-13190a5aa77541f9922dc5a2d1af6099', 'task023-7024c74bcc5244e7a2124d177ab6ec50', 'task023-9e798cbecf454bd9b1aba0149bd815dc', 'task023-67f081795bd747bd9459430e3d0002d1'] (showing up to 5)
[batch 140] tokenized len=1781, dyn_max_new_tokens=4000, tokenization_time=0.011s


[batch 140] DONE - total=17.81s gen=17.78s token_time=0.011s written=6 processed_total=1118
[batch 140] GPU reserved=16.54GB allocated=13.51GB
Done. New processed examples in this run: 1118. Output file: /content/drive/MyDrive/AdvNLP/judgemistral1.jsonl elapsed=4112.8s


In [ ]:
# recommended: latest transformers
# pip install --upgrade "git+https://github.com/huggingface/transformers.git@main"
!pip install accelerate safetensors
